# Trích xuất shot và caption trên Kaggle
Bật **Internet** và **GPU**. Notebook clone nhánh `vbs-dev`, chọn dải ID V3C, tải tối đa 10 video `<video_id>.mp4` mỗi lô từ cùng folder Drive của ASR. Pipeline tạo `scenes/V3C/<video_id>.scenes.txt` (shot general, frame đầu/cuối inclusive) và `captions/V3C/<video_id>.json` (chỉ `video_id`, `start`, `end`, `caption`). Notebook upload **chỉ JSON caption** lên folder Drive đích, một lệnh rclone mỗi lô; file shot và JSON cũng được giữ trong workspace Kaggle của phiên chạy.

OmniShotCut được giải phóng trước khi nạp VideoLLaMA3. File `.partial.json` lưu tiến độ từng caption, nên chạy lại cell cuối tiếp tục khi lỗi. Không tạo môi trường ảo; thư viện và model cache được dùng lại trong cùng phiên Kaggle. Đẩy code pipeline và notebook lên nhánh `vbs-dev` trước khi chạy.


In [ ]:
# 1. Cấu hình
import importlib.metadata, importlib.util, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # push code caption và notebook lên nhánh này trước khi chạy
DATASET = 'V3C'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
VIDEO_EXTENSION = '.mp4'
SOURCE_ROOT_FOLDER_ID = '19b7skL2b0PRJWWweWFh8qVNTLQDF6x9K'
DEST_ROOT_FOLDER_ID = '1I97-3-Y_V0-dfk-tPS4g7H8lpAHyeu2r'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
UPLOAD_BATCH_SIZE = 10
OVERWRITE_REMOTE = False

def video_id(number):
    return f'{number:05d}'

if DATASET != 'V3C' or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999:
    raise ValueError('Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999 cho V3C')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
WORK = Path('/kaggle/working/caption_pipeline')
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT = WORK / 'jobs'
JOBS_ROOT.mkdir(exist_ok=True)
STAGE_DIR = WORK / 'pending_json_7b'
STAGE_DIR.mkdir(exist_ok=True)
RCLONE_CONFIG = WORK / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = WORK / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', WORK)


In [ ]:
# 2. Clone/pull repo, cài VideoLLaMA3 + OmniShotCut và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    from collections import deque
    args = [str(item) for item in args]
    print('+', ' '.join(args[:5]), '...' if len(args) > 5 else '', flush=True)
    tail = deque(maxlen=80)
    with subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors='replace',
                          bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
        exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f'Command exited with code {exit_code}: {args[:5]}\n'
                           + ''.join(tail))
    return exit_code

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Checkout đang ở {branch}, cần {REPO_BRANCH}')
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/caption_extraction/pipeline/extract_caption.py').is_file():
    raise RuntimeError('Nhánh GitHub chưa có caption extraction; hãy push code lên vbs-dev')
print('Caption commit:', commit)

# Cài tối thiểu, giữ PyTorch GPU có sẵn trên Kaggle; không tạo venv.
required = {'transformers': '4.46.3', 'accelerate': '1.0.1'}
needs_pip = any(importlib.metadata.version(name) != version for name, version in required.items()
                if importlib.util.find_spec(name) is not None)
needs_pip |= any(importlib.util.find_spec(name) is None
                 for name in ('transformers', 'accelerate', 'decord', 'ffmpeg', 'yaml', 'cv2'))
if needs_pip:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'transformers==4.46.3', 'accelerate==1.0.1', 'decord', 'ffmpeg-python',
         'PyYAML', 'opencv-python-headless'])
if importlib.util.find_spec('omnishotcut') is None:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git'])
import yaml
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU trong Kaggle Settings')
print('GPU:', torch.cuda.get_device_name(0))

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    release = 'v1.75.2'
    archive = WORK / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-caption'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = WORK / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
run([rclone, 'version'])


In [ ]:
# 3. Liệt kê video nguồn và bỏ qua caption JSON đã upload
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_names = set(subprocess.check_output(
    [*map(str, remote_args(SOURCE_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
selected = [video_id(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}{VIDEO_EXTENSION}' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu video trong Drive nguồn: {missing}')
dest_names = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in dest_names]
print(f'Đã chọn {len(selected)} video; cần xử lý {len(pending)}:', pending)


In [ ]:
# 4. Từng lô: tải video cần xử lý, tạo shot + caption, upload JSON rồi dọn video tạm
import math
cfg = yaml.safe_load((BACKEND_DIR / 'configs/caption_extraction.yaml').read_text(encoding='utf-8'))
if cfg['extraction']['caption_model'] != 'DAMO-NLP-SG/VideoLLaMA3-7B':
    raise RuntimeError('Caption config trên nhánh này chưa đúng model')
CAPTION_ENV = os.environ.copy()
CAPTION_ENV['PYTHONPATH'] = str(BACKEND_DIR)
CAPTION_ENV['HF_HOME'] = str(WORK / 'hf_cache')
CAPTION_ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
SCENES_DIR = WORK / 'scenes' / DATASET
CAPTIONS_DIR = WORK / 'captions_7b' / DATASET
SCENES_DIR.mkdir(parents=True, exist_ok=True)
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)

def validate_json(path, vid):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list) or not data:
        raise ValueError(f'{path}: cần danh sách caption không rỗng')
    previous_start = -1.0
    for item in data:
        if (not isinstance(item, dict) or set(item) != {'video_id', 'start', 'end', 'caption'} or
                item['video_id'] != vid or not isinstance(item['caption'], str) or
                not item['caption'].strip() or
                not isinstance(item['start'], (int, float)) or
                not isinstance(item['end'], (int, float)) or
                not math.isfinite(item['start']) or not math.isfinite(item['end']) or
                not 0 <= item['start'] < item['end'] or item['start'] < previous_start):
            raise ValueError(f'{path}: segment không đúng schema')
        previous_start = item['start']
    return len(data)

# Restore staged JSON after an interrupted upload; completed files remain in CAPTIONS_DIR.
for staged in STAGE_DIR.glob('*.json'):
    local = CAPTIONS_DIR / staged.name
    if not local.is_file() or staged.read_bytes() != local.read_bytes():
        raise RuntimeError(f'JSON staged không khớp bản local: {staged}')
    staged.unlink()

# Recheck remote on cell rerun, so completed batches are not uploaded again.
remote_done = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
remaining = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in remote_done]
completed = {}
for batch_number, start in enumerate(range(0, len(remaining), UPLOAD_BATCH_SIZE)):
    batch_ids = remaining[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{batch_number:04d}'
    video_dir = batch_work / 'videos' / DATASET
    video_dir.mkdir(parents=True, exist_ok=True)
    need_processing = [vid for vid in batch_ids if not (CAPTIONS_DIR / f'{vid}.json').is_file()]
    for vid in need_processing:
        path = video_dir / f'{vid}{VIDEO_EXTENSION}'
        if not path.is_file() or path.stat().st_size == 0:
            run([*remote_args(SOURCE_ROOT_FOLDER_ID), 'copyto',
                 f'{RCLONE_REMOTE}:{path.name}', path])
        else:
            print('Dùng lại video đã tải:', path.name)
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Video tải về rỗng: {path.name}')
    if need_processing:
        cmd = [sys.executable, '-m', 'scripts.caption.extract.run', '--dataset', DATASET.lower(),
               '--input-dir', video_dir, '--scenes-dir', SCENES_DIR, '--output-dir', CAPTIONS_DIR]
        for vid in need_processing:
            cmd += ['--video-id', vid]
        run(cmd, cwd=BACKEND_DIR, env=CAPTION_ENV)
    for vid in batch_ids:
        result = CAPTIONS_DIR / f'{vid}.json'
        if not result.is_file():
            raise FileNotFoundError(f'Không có caption JSON: {result}')
        completed[vid] = validate_json(result, vid)
        shutil.copy2(result, STAGE_DIR / result.name)
        print(f'Caption xong {vid}: {completed[vid]} segment')
    for path in video_dir.glob(f'*{VIDEO_EXTENSION}'):
        path.unlink()  # giữ shot/JSON trong WORK, giải phóng video trước khi upload
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(batch_ids))), '--checkers', '10'])
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in STAGE_DIR.glob('*.json'):
        path.unlink()
    shutil.rmtree(batch_work)
    print(f'Đã upload {len(batch_ids)} caption JSON; giữ shot/JSON local, dọn video tạm')
print('Hoàn tất:', completed, '| commit:', commit)
